# 🔮 KuroNeko TR - Phi-4 Mini Türkçe Eğitim

**Platform:** Colab T4 GPU  
**Model:** microsoft/Phi-4-mini-instruct (3.8B)  
**Yöntem:** QLoRA (4-bit)  
**Hedef:** Türkçe sohbet + akıl yürütme + kod üretimi  
**Veri:** 25K+ Türkçe örnek

---

**Nasıl çalıştırılır:**
1. Runtime → Change runtime type → T4 GPU
2. HF Token gir (sonra)
3. Run All (Tümünü Çalıştır)
4. Eğitimin bitmesini bekle (3-4 saat)


In [ ]:
# ── 1. KURULUM ──────────────────────────────────────────────────────────────
import subprocess, sys, os

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-500:])
    print(result.stdout[-300:])

# Temiz kurum - önce eski paketleri temizle
print('📦 Paketler kuruluyor...')

# Adım 1: Temel bağımlılıklar
run('pip install -q --upgrade pip')
run('pip install -q --upgrade huggingface_hub>=0.25.0')
run('pip install -q --upgrade transformers>=4.45.0')
run('pip install -q --upgrade peft>=0.12.0')
run('pip install -q --upgrade bitsandbytes>=0.44.0')
run('pip install -q --upgrade accelerate>=0.34.0')
run('pip install -q datasets>=2.20.0')

# Adım 2: PyTorch (Colab'da genelde hazır, gürse güncelle)
run('pip install -q --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121')

# Adım 3: TRL (SFT için)
run('pip install -q trl>=0.11.0')

print('✅ Kurulum tamamlandı')

In [ ]:
# ── 2. HF TOKEN ─────────────────────────────────────────────────────────────
import os

HF_TOKEN = ''  #@param {type:'string'}
os.environ['HF_TOKEN'] = HF_TOKEN

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print('✅ HF login OK')
else:
    print('⚠️ HF_TOKEN boş - model push yapılmayacak')

In [ ]:
# ── 3. GPU KONTROL ──────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0)) / 1e9
    print(f'✅ VRAM: {vram:.1f} GB')
    print(f'✅ CUDA: {torch.version.cuda}')
else:
    print('⚠️ GPU bulunamadı! Runtime → Change runtime type → T4 GPU')
    print('Sonra Run All tekrar çalıştır.')

In [ ]:
# ── 4. VERİ HAZIRLAMA (25K+ Türkçe) ────────────────────────────────────────
from datasets import load_dataset, Dataset, concatenate_datasets
import random, json, os

random.seed(42)
os.makedirs('data/processed', exist_ok=True)

all_data = []

def add_sample(instruction, output, input_text=''):
    if instruction and output and len(instruction) > 10 and len(output) > 10:
        all_data.append({
            'instruction': instruction[:1000],
            'input': input_text[:500],
            'output': output[:1000]
        })

def safe_load(ds_name, col_q, col_a, col_ctx=None, limit=8000):
    try:
        ds = load_dataset(ds_name, split='train', cache_dir='/content/hf_cache')
        if len(ds) > limit:
            ds = ds.shuffle(seed=42).select(range(limit))
        cnt = 0
        for row in ds:
            q = str(row.get(col_q, '')).strip()
            a = str(row.get(col_a, '')).strip()
            ctx = ''
            if col_ctx and row.get(col_ctx):
                ctx = str(row[col_ctx]).strip()
            add_sample(q, a, ctx)
            cnt += 1
        print(f'  ✅ {ds_name}: {cnt}')
    except Exception as e:
        print(f'  ⚠️ {ds_name}: {str(e)[:80]}')

print('📥 HuggingFace veri setleri yükleniyor...')
print('=' * 50)

# Türkçe veri setleri (yüksek öncelik)
safe_load('merve/turkish_instructions', 'instruction', 'output', 'input', 8000)
safe_load('ucekmez/OpenOrca-tr', 'question', 'response', None, 8000)
safe_load('atasoglu/databricks-dolly-15k-tr', 'instruction', 'response', 'context', 8000)
safe_load('ytu-ce-cosmos/gsm8k_tr', 'question', 'answer', None, 5000)
safe_load('beratcmn/no_robots_turkish', 'instruction', 'output', None, 5000)
safe_load('beratcmn/lima-tr', 'instruction', 'output', None, 3000)
safe_load('emre/stanford-alpaca-cleaned-turkish-translated', 'instruction', 'output', 'input', 5000)
safe_load('alibayram/turkish_mmlu', 'question', 'answer', None, 5000)
safe_load('nisancoskun/turkish_general_knowledge_qa', 'question', 'answer', None, 5000)

print(f'\n📦 HF veri setleri tamamlandı: {len(all_data)} örnek')


In [ ]:
# ── 4B. SENTETIK VERİ (10K+ Türkçe) ────────────────────────────────────────

print('🔧 Sentetik Türkçe veri oluşturuluyor...')

# Matematik / Akıl Yürütme (200+)
math_templates = [
    ('Bir markette {a} TL/kg olan elma ile {b} TL/kg olan armut alıyorum. {c} kg elma ve {d} kg armut için kaç TL öderim?',
     lambda a,b,c,d: f'Adım adım:\nElma: {a}×{c}={a*c} TL\nArmut: {b}×{d}={b*d} TL\nToplam: {a*c+b*d} TL'),
    ('Bir dikdörtgenin uzun kenarı {a} cm, kısa kenarı {b} cm. Çevresi ve alanı kaç?',
     lambda a,b: f'Çevre: 2×({a}+{b})={2*(a+b)} cm\nAlan: {a}×{b}={a*b} cm²'),
    ('Bir sayının %{p} si {r} ise bu sayı kaçtır?',
     lambda p,r: f'x × {p}/100 = {r}\nx = {r}/({p}/100) = {r/(p/100):.0f}'),
    ('Bir araba {h1} saatte {s1} km, sonra {h2} saatte {s2} km yol aldı. Toplam kaç km?',
     lambda h1,s1,h2,s2: f'İlk: {h1}×{s1}={h1*s1} km\nİkinci: {h2}×{s2}={h2*s2} km\nToplam: {h1*s1+h2*s2} km'),
    ('Bir sınıfta {t} öğrenci var. %{k} kız. Kız öğrenci sayısı kaç?',
     lambda t,k: f'{t} × {k}/100 = {t*k//100} kız öğrenci'),
]

import random as rnd
rnd.seed(42)

for _ in range(500):
    template, calc_fn = rnd.choice(math_templates)
    try:
        params = [rnd.randint(3, 50) for _ in range(template.count('{'))]
        q = template.format(*params)
        a = calc_fn(*params)
        add_sample(q, a)
    except:
        pass

# Kod üretim (300+)
code_examples = [
    ('Python\'da bir liste nasıl sıralanır?', 'Python\'da liste sıralama:\n\n```python\n# sorted() - yeni liste döndürür\nters = sorted(liste, reverse=True)\n\n# sort() - yerinde sıralar\nliste.sort()\nliste.sort(reverse=True)\n\n# Anahtara göre sıralama\nogs = [{"ad": "Ali", "yas": 30}, {"ad": "Ayşe", "yas": 25}]\nsirali = sorted(ogs, key=lambda x: x["yas"])\n```'),
    ('Python\'da bir dosyayı nasıl okursun?', '```python\n# Yöntem 1: with (önerilen)\nwith open("dosya.txt", "r", encoding="utf-8") as f:\n    icerik = f.read()\n\n# Yöntem 2: satır satır\nwith open("dosya.txt") as f:\n    for satir in f:\n        print(satir.strip())\n\n# Yöntem 3: liste olarak\nsatirlar = open("dosya.txt").readlines()\n```'),
    ('Python\'da bir sözlük (dict) nasıl oluşturulur?', '```python\n# Boş sözlük\nd = {}\nd = dict()\n\n# Değerlerle\nd = {"ad": "Ali", "yas": 30, "sehir": "Ankara"}\n\n# Erişim\nprint(d["ad"])  # Ali\nprint(d.get("olmayan", "varsayılan"))\n\n# Ekleme/Güncelleme\nd["meslek"] = "Mühendis"\n\n# Döngü\nfor anahtar, deger in d.items():\n    print(f"{anahtar}: {deger}")\n```'),
    ('Git\'te commit nasıl atılır?', '```bash\n# Değişikliği sahneye ekle\ngit add dosya.txt\n# Veya tüm değişiklikler\ngit add .\n\n# Commit at\ngit commit -m "Açıklama mesajı"\n\n# Push et\ngit push origin main\n\n# Hızlı commit (değişiklikleri ekle + commit)\ngit commit -am "Hızlı commit"\n```'),
    ('Docker container\'ini arka planda çalıştırmak için komut nedir?', '```bash\n# Arka planda çalıştır\ndocker run -d --name konteyner_adi image_adi\n\n# Port yönlendirme ile\ndocker run -d -p 8080:80 --name web nginx\n\n# Volume mount ile\ndocker run -d -v /host:/container --name app image\n\n# Komut gönder\ndocker exec konteyner_adi komut\n# Container\'ı durdur\ndocker stop konteyner_adi\n# Sil\ndocker rm konteyner_adi\n```'),
    ('SQL injection nedir, nasıl önlenir?', 'SQL injection, saldırganın SQL sorgularına zararlı kod enjekte etmesidir.\n\n**Önleme:**\n```python\n# KÖTÜ - String birleştirme\ncursor.execute(f"SELECT * FROM users WHERE id = {user_id}")\n\n# İYİ - Parametreli sorgu\ncursor.execute("SELECT * FROM users WHERE id = %s", (user_id,))\n\n# EN İYİ - ORM kullan\nuser = session.query(User).filter(User.id == user_id).first()\n```'),
    ('Python\'da bir API\'ye HTTP isteği nasıl atılır?', '```python\nimport requests\n\n# GET isteği\nresponse = requests.get("https://api.example.com/users")\nprint(response.json())\n\n# POST isteği\ndata = {"ad": "Ali", "yas": 30}\nresponse = requests.post("https://api.example.com/users", json=data)\n\n# Headers ile\nheaders = {"Authorization": "Bearer TOKEN"}\nresponse = requests.get("https://api.example.com/data", headers=headers)\n\n# Hata yönetimi\ntry:\n    response.raise_for_status()\nexcept requests.exceptions.HTTPError as e:\n    print(f"Hata: {e}")\n```'),
    ('Linux\'ta bir dosyanın izinlerini nasıl değiştiririm?', '```bash\n# Sayısal\nchmod 755 dosya.sh    # rwxr-xr-x\nchmod 644 dosya.txt   # rw-r--r--\n\n# Harfli\nchmod +x dosya.sh     # Çalıştırabilir\nchmod u+w dosya.txt   # Yazma izni\nchmod g-r dosya.txt   # Grup okumayı kaldır\n\n# İzin değerleri\n# 4: Okuma (r), 2: Yazma (w), 1: Çalıştırma (x)\n\n# Recursive\nchmod -R 755 klasor/\n```'),
]

for instr, out in code_examples * 20:
    add_sample(instr, out)

# Türkçe kültür & genel kültür (200+)
culture_examples = [
    ('Türkiye\'nin başkenti neresidir?', 'Türkiye\'nin başkenti Ankara\'dır. 13 Ekim 1923\'te başkent ilan edilmiştir. Nüfusu yaklaşık 6 milyon olan Ankara, Türkiye\'nin ikinci en büyük şehridir.'),
    ('Atatürk kimdir?', 'Mustafa Kemal Atatürk (1881-1938), Türkiye Cumhuriyeti\'nin kurucusu ve ilk Cumhurbaşkanıdır. Kurtuluş Savaşı\'nı önderleyerek bağımsız Türkiye\'yi kurmuş, çağdaşlaşma reformları yapmıştır.'),
    ('Cumhuriyet ne zaman ilan edildi?', 'Türkiye Cumhuriyeti 29 Ekim 1923\'te ilan edilmiştir.'),
    ('Osmanlı İmparatorluğu ne zaman kuruldu?', 'Osmanlı İmparatorluğu 1299\'da Osman Bey tarafından kurulmuştur. 1453\'te İstanbul\'un fethiyle büyük güce ulaşmış, 1922\'de sona ermiştir.'),
    ('Türkiye\'nin en büyük 5 şehri nedir?', 'Türkiye\'nin en büyük şehirleri:\n1. İstanbul (~16 milyon)\n2. Ankara (~6 milyon)\n3. İzmir (~4.5 milyon)\n4. Bursa (~3 milyon)\n5. Antalya (~2.5 milyon)'),
    ('Türk mutfağının önemli lezzetleri nelerdir?', 'Türk mutfağının önemli lezzetleri:\n\n**Ana Yemekler:** Kebap çeşitleri (Adana, Urfa, İskender), köfte, karnıyarık, mantı, lahmacun, etli nohut.\n\n**Tatlılar:** Baklava, künefe, katmer, lokum, helva, sütlaç.\n\n**İçecekler:** Türk kahvesi, çay, ayran.'),
    ('Türkiye\'nin UNESCO Dünya Mirası listesindeki yerleri nelerdir?', 'Türkiye\'nin UNESCO Dünya Mirası:\n1. Kapadokya (Nevşehir)\n2. Pamukkale (Denizli)\n3. Efes (İzmir)\n4. Nemrut Dağı (Adıyaman)\n5. Safranbolu (Karabük)\n6. İstanbul Tarihi Alanlar\n7. Göbeklitepe (Şanlıurfa)\n8. Sümela Manastırı (Trabzon)'),
    ('Fotosentez nedir?', 'Fotosentez, bitkilerin güneş ışığını kullanarak karbondioksit ve sudan glikoz ve oksijen üretme sürecidir.\n\nFormül: Su + CO₂ + Güneş ışığı → Şeker + Oksijen\n\nBitkilerin yapraklarındaki klorofil bu süreci gerçekleştirir.'),
    ('DNA nedir?', 'DNA (Deoksiribonükleik Asit), canlıların genetik bilgisini taşıyan çift sarmal moleküldür. Dört baz içerir: Adenin (A), Timin (T), Guanın (G), Sitozin (C).'),
    ('Güneş sistemi nedir?', 'Güneş sistemi, Güneş ve etrafında dönen 8 gezegen içerir:\n1. Merkür, 2. Venüs, 3. Dünya, 4. Mars, 5. Jüpiter, 6. Satürn, 7. Uranüs, 8. Neptün'),
    ('I. Dünya Savaşı ne zaman başladı?', 'I. Dünya Savaşı 28 Temmuz 1914\'te başladı, 11 Kasım 1918\'de bitti. 17 milyon ölüm oldu.'),
    ('II. Dünya Savaşı ne zaman başladı?', 'II. Dünya Savaşı 1 Eylül 1939\'da başladı, 2 Eylül 1945\'te bitti. 70-85 milyon ölüm oldu.'),
    ('Kuantum bilgisayar nedir?', 'Kuantum bilgisayar, kuantum mekaniği ilkelerini kullanır. Klasik bit yerine qubit kullanır. Süperpozisyon ve dolanıklık sayesinde çok daha hızlı hesaplama yapar.'),
    ('Blockchain nedir?', 'Blockchain, dağıtık ve değiştirilemez bir veritabanıdır. Her blok önceki bloğa hash ile bağlanır. Kripto para, NFT, akıllı kontrat gibi alanlarda kullanılır.'),
    ('Yapay zeka nedir?', 'Yapay zeka, makinelerin insan benzeri zeka göstermesini sağlayan teknolojidir. Makine öğrenmesi, derin öğrenme, NLP, bilgisayar görüsü alt dallarıdır.'),
]

for instr, out in culture_examples * 15:
    add_sample(instr, out)

# Sohbet (500+)
sohbet_examples = [
    ('Merhaba, nasılsın?', 'Merhaba! İyiyim, teşekkür ederim. Size nasıl yardımcı olabilirim?'),
    ('Teşekkürler!', 'Rica ederim! Başka bir sorunuz varsa yardımcı olmaktan mutluluk duyarım.'),
    ('Görüşürüz!', 'Görüşmek üzere! İyi günler dilerim.'),
    ('Yardım edebilir misin?', 'Tabii ki! Ne konuda yardıma ihtiyacınız var?'),
    ('Bu konuda bilgin var mı?', 'Evet, bu konuda size yardımcı olabilirim. Sorunuzu detaylandırır mısınız?'),
    ('Sen kimsin?', 'Ben KuroNeko TR, Türkçe konuşan bir yapay zeka asistanıyım. Sorularınıza yardımcı olmak için buradayım.'),
    ('Ne yapabilirsin?', 'Sorularınıza yanıt verebilir, kod yazabilir, metinler oluşturabilir, çeviri yapabilir ve daha birçok konuda yardımcı olabilirim.'),
    ('Python nedir?', 'Python, 1991\'de Guido van Rossum tarafından geliştirilen yüksek seviyeli bir programlama dilidir. Sade sözdizimi, geniş ekosistem ve çok amaçlı kullanımıyla popülerdir.'),
    ('JavaScript nedir?', 'JavaScript, web geliştirme için kullanılan bir programlama dilidir. Hem tarayıcıda hem sunucu tarafta (Node.js) çalışır.'),
    ('Git nedir?', 'Git, yazılım geliştirme sırasında kod değişikliklerini takip eden bir sürüm kontrol sistemidir.'),
]

for instr, out in sohbet_examples * 40:
    add_sample(instr, out)

# Dizi/Örüntü (200+)
for _ in range(200):
    start = rnd.randint(1, 20)
    step = rnd.randint(2, 5)
    seq = [start + step*i for i in range(5)]
    q = f"{', '.join(map(str, seq[:-1]))}, ? dizisinde sonraki sayı kaçtır?"
    a = f"Bu dizi her adımda {step} ekleyerek ilerler.\nSonraki sayı: {seq[-1]}"
    add_sample(q, a)

print(f'\n📊 Sentetik veri tamamlandı')
print(f'📊 TOPLAM: {len(all_data)} örnek')

# Kaydet
random.shuffle(all_data)
val_size = int(len(all_data) * 0.05)
val_data = all_data[:val_size]
train_data = all_data[val_size:]

with open('data/processed/train.jsonl', 'w', encoding='utf-8') as f:
    for item in train_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

with open('data/processed/val.jsonl', 'w', encoding='utf-8') as f:
    for item in val_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f'✅ Train: {len(train_data)} örnek')
print(f'✅ Val: {len(val_data)} örnek')

In [ ]:
# ── 5. EĞİTİM (3 epoch) ─────────────────────────────────────────────────────
import time, os

print('🚀 Eğitim başlıyor...')
print(f'📊 Eğitim verisi: {len(train_data)} örnek')
print(f'📊 Epoch: 3')
print(f'📊 GPU: {torch.cuda.get_device_name(0)}')
print('=' * 50)

t0 = time.time()

# Unsloth ile hızlı eğitim (önerilen)
# Önce Unsloth kur
try:
    import unsloth
    print('✅ Unsloth zaten kurulu')
except ImportError:
    print('📦 Unsloth kuruluyor...')
    subprocess.run('pip install -q unsloth', shell=True)

# Unsloth ile eğitim
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='microsoft/Phi-4-mini-instruct',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
    token=HF_TOKEN,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

print('✅ Model yüklendi, eğitim başlıyor...')

# Eğitim
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

train_ds = Dataset.from_list(train_data)
val_ds = Dataset.from_list(val_data)

def format_fn(examples):
    texts = []
    for i in range(len(examples['instruction'])):
        q = examples['instruction'][i]
        a = examples['output'][i]
        inp = examples['input'][i] if examples['input'][i] else ''
        if inp:
            q = f"{q}\n{inp}"
        text = f"<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{a}<|im_end|>"
        texts.append(text)
    return {'text': texts}

train_ds = train_ds.map(format_fn, batched=True)
val_ds = val_ds.map(format_fn, batched=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field='text',
    max_seq_length=2048,
    args=TrainingArguments(
        output_dir='/content/output',
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        warmup_steps=50,
        logging_steps=50,
        save_steps=200,
        optim='adamw_8bit',
        lr_scheduler_type='cosine',
        report_to='none',
        seed=42,
        fp16=True,
        evaluation_strategy='steps',
        eval_steps=200,
        load_best_model_at_end=True,
    ),
)

result = trainer.train()

elapsed = (time.time() - t0) / 60
print(f'\n✅ Eğitim tamamlandı! Süre: {elapsed:.1f} dakika')
print(f'📊 Son loss: {result.training_loss:.4f}')

In [ ]:
# ── 6. MODEL TESTİ ──────────────────────────────────────────────────────────
print('🧪 MODEL TESTİ')
print('=' * 60)

FastLanguageModel.for_inference(model)

test_questions = [
    'Merhaba! Kendini tanıtır mısın?',
    'Türkiye\'nin başkenti neresidir?',
    'Python\'da bir liste nasıl sıralanır? Örnek ver.',
    'Fotosentez nedir? Basit bir dille açıkla.',
    'Bir markette elma 5 TL/kg, armut 7 TL/kg. 3 kg elma ve 2 kg armut alsam toplam kaç TL öderim?',
    'Docker container\'ini arka planda çalıştırmak için komut nedir?',
    'Git\'te bir branch\'i silmek için komut nedir?',
    'SQL injection nedir, nasıl önlenir?',
]

for q in test_questions:
    prompt = f'<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n'
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\n❓ {q}')
    print(f'🤖 {response.strip()}')
    print('-' * 40)

In [ ]:
# ── 7. PUSH TO HUB ──────────────────────────────────────────────────────────
HUB_ID = 'KuroNeko1234t/phi4-mini-tr'
HF_TOKEN = os.environ.get('HF_TOKEN', '')

if HF_TOKEN:
    # Kaydet
    model.save_pretrained('/content/merged')
    tokenizer.save_pretrained('/content/merged')
    print('✅ Model kaydedildi')
    
    # Push
    from huggingface_hub import create_repo
    create_repo(HUB_ID, token=HF_TOKEN, exist_ok=True, repo_type='model')
    model.push_to_hub(HUB_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(HUB_ID, token=HF_TOKEN)
    print(f'\n✅ Model yüklendi: https://huggingface.co/{HUB_ID}')
else:
    print('\n⚠️ HF_TOKEN yok, push atlandı')
    print(f'Model /content/merged dizinine kaydedildi')

print('\n🎉 TÜM AŞAMALAR TAMAMLANDI!')
print(f'Model: huggingface.co/{HUB_ID}')
print('Space: huggingface.co/spaces/KuroNeko1234t/phi4-mini-tr')